In [0]:
CREATE OR REPLACE PROCEDURE gold.run_all()
SQL SECURITY INVOKER
BEGIN

  -- 1. Load dimensions
  CALL gold.load_dim_coin();
  CALL gold.load_dim_date();

  -- 2. Load facts
  CALL gold.load_fact_prices();
  CALL gold.load_fact_ohlc();

  -- 3. Optimize tables
  OPTIMIZE gold.fact_prices ZORDER BY (coin_id, price_timestamp);
  OPTIMIZE gold.fact_ohlc ZORDER BY (coin_id, date);

  -- 4. Clean old files
  VACUUM gold.fact_prices RETAIN 168 HOURS;
  VACUUM gold.fact_ohlc RETAIN 168 HOURS;

  -- 5. Updating etl_log - fact_prices
  INSERT INTO gold.etl_log
  VALUES (current_timestamp(), 'load_fact_prices', 'SUCCESS',
        (SELECT COUNT(*) FROM gold.fact_prices), NULL);
        
   -- 6. Updating etl_log - fact_ohlc
  INSERT INTO gold.etl_log
  VALUES (current_timestamp(), 'load_fact_ohlc', 'SUCCESS',
        (SELECT COUNT(*) FROM gold.fact_ohlc), NULL);

END;